# Lightweight IDS for resource-constrained Edge IoT devices
**Project overview.** 

This notebook develops and evaluates machine learning models that detect cyberattacks in IoT network traffic, using the network portion of the TON_IoT dataset (UNSW), which contains Zeek-style flow records from an IIoT testbed labeled as normal or as one of several attack types. The data is first explored and cleaned, after which baseline models (logistic regression, random forest and LightGBM) are trained for binary detection (attack vs. normal) and then for multiclass attack typing. Because the target setting is resource-constrained edge devices, a full feature set is compared against a reduced set of features that could realistically be computed on-device, and model size and inference speed are measured alongside accuracy. Throughout, the validity of high scores is tested by checking for leakage (identifier columns, duplicates, grouped data), and performance is reported using precision, recall, macro-F1 and PR-AUC rather than plain accuracy. The notebook closes with findings and limitations, including that the data originates from a simulated testbed and has not been validated on live traffic.

In [1]:
import pandas as pd
import numpy as np

file_path = '/kaggle/input/datasets/j21saksham/ton-iot-network-intrusion-detection/train_test_network.csv'
df = pd.read_csv(file_path)
df.shape
print(df.head())
print(df.info())

          src_ip  src_port         dst_ip  dst_port proto service    duration  \
0   192.168.1.37      4444  192.168.1.193     49178   tcp       -  290.371539   
1  192.168.1.193     49180   192.168.1.37      8080   tcp       -    0.000102   
2  192.168.1.193     49180   192.168.1.37      8080   tcp       -    0.000148   
3  192.168.1.193     49180   192.168.1.37      8080   tcp       -    0.000113   
4  192.168.1.193     49180   192.168.1.37      8080   tcp       -    0.000130   

   src_bytes  dst_bytes conn_state  ...  http_response_body_len  \
0     101568       2592        OTH  ...                       0   
1          0          0        REJ  ...                       0   
2          0          0        REJ  ...                       0   
3          0          0        REJ  ...                       0   
4          0          0        REJ  ...                       0   

   http_status_code  http_user_agent  http_orig_mime_types  \
0                 0                -            

## Data Exploration
The dataset has 44 columns and 211,043 rows. 
27 columns are 'object', 16 are 'int64' and 1 is 'float64'.

### Observations
- No nulls reported, but '-' is used as a placeholder as shown by info(). Hidden missing values in some rows.
- File appears to be grouped by attack type. First 5 rows show 'backdoor'
- Two label columns. Both type and label seem to encode the same info (label appears 1 for every backdoor type).
- Identifier columns present may be associated with certain attacks. Model might recognize the testbed, not attack type.


In [2]:
raw = df.copy()
df = raw.copy()

Created a working copy for manipulation without harming original dataset.

### Label vs Type

In [3]:
df["label"].value_counts()
df["label"].value_counts(normalize=True)
df["type"].value_counts()
pd.crosstab(df["label"], df["type"])

type,backdoor,ddos,dos,injection,mitm,normal,password,ransomware,scanning,xss
label,,,,,,,,,,
0,0,0,0,0,0,50000,0,0,0,0
1,20000,20000,20000,20000,1043,0,20000,20000,20000,20000


Label and type carry overlapping information. Label is the binary version (0 = normal, 1 = attack) while type is the detailed version. 
Both are targets with their own use cases. For binary classification, use label. Multiclass classifications will rely on type.

### File Ordering

In [4]:
print(df["type"].head(20).value_counts())
print(df["type"].iloc[-20:].value_counts())
print(df["type"].ne(df["type"].shift()).sum())

type
backdoor    20
Name: count, dtype: int64
type
xss    20
Name: count, dtype: int64
10


The first 20 rows are all `backdoor` and the last 20 are all `xss`. The `type` column changes value only 10 times, i.e. each of the 10 types forms one contiguous block. The file is therefore sorted by attack type, and splits must be stratified random rather than positional.

### Duplicates

In [5]:
print(df.duplicated().sum())
print(df.duplicated().mean())
print(df.drop_duplicates()["type"].value_counts())

20569
0.09746355008221073
type
normal        42040
scanning      20000
ddos          19993
injection     19964
password      19861
dos           18992
backdoor      18711
xss           15137
ransomware    14735
mitm           1041
Name: count, dtype: int64


About 9.7% of the rows (20,569 of 211,043) are exact duplicates. They are very unevenly spread across classes, which matters more than the total.

They are unevenly distributed: ransomware (-5,265) and xss (-4,863) lose about a quarter of their rows, normal loses 7,960 (16%), backdoor and dos lose around 5-6%, and the remaining classes lose under 1%. `scanning` has none. After deduplication the per-class counts are no longer uniform, and the dataset has 190,474 rows (about 78% attack, 22% normal). `mitm` remains the smallest class (1,041 rows). Deduplication is applied before any train/test split to avoid identical rows appearing on both sides.

### Hidden missing values

In [6]:
dash = (df == "-").mean().sort_values(ascending=False)
dash[dash>0]

ssl_issuer              0.999948
ssl_subject             0.999948
http_orig_mime_types    0.999924
weird_addl              0.999256
http_resp_mime_types    0.999033
http_user_agent         0.998640
http_uri                0.998640
http_method             0.998640
http_version            0.998588
http_trans_depth        0.998564
weird_notice            0.998313
weird_name              0.998313
ssl_established         0.998100
ssl_resumed             0.998100
ssl_version             0.998100
ssl_cipher              0.998100
dns_query               0.834891
dns_rejected            0.834095
dns_AA                  0.834095
dns_RD                  0.834095
dns_RA                  0.834095
service                 0.625617
dtype: float64

Although `info()` reports no nulls, 21 columns use `-` as a placeholder. All `ssl_*`, `http_*` and `weird_*` columns are 99.8-99.99% placeholders (as few as ~11 populated rows for `ssl_issuer`/`ssl_subject`). The `dns_*` columns are ~83.4% placeholders, and `service` is 62.6%. 

In Zeek logs these values mean the protocol was absent from the flow rather than that data was lost, so protocol presence may itself be informative.

In [7]:
cols = ["http_method", "ssl_version", "dns_query", "weird_name", "service"]
for c in cols:
    print(c)
    print((df[c] != "-").groupby(df["type"]).mean().round(4))

http_method
type
backdoor      0.0000
ddos          0.0007
dos           0.0000
injection     0.0001
mitm          0.0000
normal        0.0047
password      0.0016
ransomware    0.0000
scanning      0.0000
xss           0.0002
Name: http_method, dtype: float64
ssl_version
type
backdoor      0.0000
ddos          0.0001
dos           0.0000
injection     0.0000
mitm          0.0000
normal        0.0080
password      0.0000
ransomware    0.0000
scanning      0.0000
xss           0.0000
Name: ssl_version, dtype: float64
dns_query
type
backdoor      0.0000
ddos          0.2570
dos           0.0737
injection     0.0352
mitm          0.5292
normal        0.4965
password      0.0088
ransomware    0.0000
scanning      0.0288
xss           0.0699
Name: dns_query, dtype: float64
weird_name
type
backdoor      0.0000
ddos          0.0001
dos           0.0000
injection     0.0000
mitm          0.0000
normal        0.0071
password      0.0000
ransomware    0.0000
scanning      0.0000
xss           0.

`http_*`, `ssl_*` and `weird_*` are populated in well under 1% of rows (roughly 290-400 rows each), and nearly all of those rows are `normal`. They carry no usable signal and require application-layer parsing, so they should be dropped. 

Notably, `injection` and `xss` (web attacks) have almost no HTTP fields, which suggests the HTTP log was not captured for those flows. 

In contrast, DNS and `service` vary strongly by class: `dns_query` is populated for 0% of backdoor and ransomware flows but ~50% of normal and mitm flows, and `service` ranges from <1% (backdoor) to 88% (injection). These are kept as presence flags / categorical levels (`service = "-"` as its own category). Because such class-dependent patterns may reflect testbed artifacts, their contribution is re-tested in the leakage check.